In [15]:
import pickle
import pandas as pd

with open("final_rf_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

In [16]:
from datetime import datetime

user_input = input("Enter date and time (YYYY-MM-DD HH:MM:SS): ")

try:
    dt = datetime.strptime(user_input, "%Y-%m-%d %H:%M:%S")
    print("You entered:", dt)
except ValueError:
    print("Incorrect format! Please use YYYY-MM-DD HH:MM:SS")


You entered: 2025-10-20 10:00:00


In [17]:
import holidays

df = pd.DataFrame([{'datetime': dt}])

# Extract features
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['date'] = df['datetime'].dt.day
df['day'] = df['datetime'].dt.dayofweek       
df['is_weekend'] = df['day'].isin([5, 6]).astype(int)

indian_holidays = holidays.India(years=range(2021, 2035))
df['is_holiday'] = df['datetime'].dt.normalize().isin(indian_holidays).astype(int)

print(df[['year', 'month', 'date', 'day', 'is_weekend','is_holiday']])

   year  month  date  day  is_weekend  is_holiday
0  2025     10    20    0           0           1


   year  month  date  day  is_weekend  is_holiday
0  2025     10    20    0           0           1


C:\Users\hanam\AppData\Local\Temp\ipykernel_15244\2279364372.py:13: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday'] = df['datetime'].dt.normalize().isin(indian_holidays).astype(int)


In [18]:
dt_date = dt.date()

indian_holidays = holidays.India(years=range(2021, 2035))
    
if dt_date in indian_holidays:
    is_festival = 1
    festival_name = indian_holidays[dt_date]
    
    # Find same festival last year
    last_year = dt.year - 1
    festival_last_year_date = None
    for date, name in indian_holidays.items():
        if name == festival_name and date.year == last_year:
            festival_last_year_date = date
            break
    
    if festival_last_year_date:
        festival_day_last_year = festival_last_year_date.day
        festival_month_last_year = festival_last_year_date.month
    else:
        festival_day_last_year = -1
        festival_month_last_year = -1
else:
    is_festival = 0
    festival_name = "No Festival"
    festival_day_last_year = -1
    festival_month_last_year = -1

df['is_festival'] = is_festival
df['festival_name'] = festival_name
df['festival_day_last_year'] = festival_day_last_year
df['festival_month_last_year'] = festival_month_last_year

print(df)

             datetime  year  month  date  day  is_weekend  is_holiday  \
0 2025-10-20 10:00:00  2025     10    20    0           0           1   

   is_festival festival_name  festival_day_last_year  festival_month_last_year  
0            1        Diwali                      31                        10  


In [19]:
import pickle
import pandas as pd

# Load encoder and transform festival_name
with open("ordinal_encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

df['festival_name'] = encoder.transform(df[['festival_name']])

df

,datetime,year,month,date,day,is_weekend,is_holiday,is_festival,festival_name,festival_day_last_year,festival_month_last_year
0,2025-10-20 10:00:00,2025,10,20,0,0,1,1,-1.0,31,10


In [20]:
# Check if there was a festival on the same date last year
from datetime import timedelta

# Calculate the date exactly 1 year ago
dt_last_year = dt - timedelta(days=365)
dt_last_year_date = dt_last_year.date()

print(f"📅 User input date: {dt.date()}")
print(f"📅 Same date last year: {dt_last_year_date}")

# Check if last year's date was a festival
indian_holidays = holidays.India(years=range(2021, 2035))

if dt_last_year_date in indian_holidays:
    is_festival_last_year = 1
    festival_name_last_year = indian_holidays[dt_last_year_date]
    print(f"✅ Last year it was a festival: {festival_name_last_year}")
else:
    is_festival_last_year = 0
    festival_name_last_year = "No Festival"
    print(f"❌ Last year was NOT a festival")

# Add to dataframe
df['is_festival_last_year'] = is_festival_last_year

print(f"\n📊 Festival Status:")
print(f"   This year ({dt.date()}): {'Festival' if df['is_festival'].iloc[0] == 1 else 'Not a Festival'}")
print(f"   Last year ({dt_last_year_date}): {'Festival' if is_festival_last_year == 1 else 'Not a Festival'}")

print("\n✅ Updated DataFrame:")
print(df[['datetime', 'is_festival', 'festival_name', 'is_festival_last_year']])
# Transform using the previously fitted encoder
df['festival_name'] = encoder.transform(df[['festival_name']])

print(df)


📅 User input date: 2025-10-20
📅 Same date last year: 2024-10-20
❌ Last year was NOT a festival

📊 Festival Status:
   This year (2025-10-20): Festival
   Last year (2024-10-20): Not a Festival

✅ Column 'is_festival_last_year' added successfully!

Current festival_name (before encoding): -1.0


,datetime,year,month,date,day,is_weekend,is_holiday,is_festival,festival_name,festival_day_last_year,festival_month_last_year,is_festival_last_year
0,2025-10-20 10:00:00,2025,10,20,0,0,1,1,-1.0,31,10,0


In [21]:
from datetime import datetime, timedelta
import requests
import pandas as pd

def get_temperature_delhi_final(target_dt):
    """
    Fetch temperature for Delhi at the given datetime.
    - Past dates (older than yesterday): Uses Open-Meteo Historical API
    - Recent dates (within 7 days): Uses Open-Meteo Forecast API
    - Far future dates: Uses historical monthly averages
    """
    lat, lon = 28.7041, 77.1025  # Delhi coordinates
    now = datetime.now()
    yesterday = now - timedelta(days=1)
    
    # CASE 1: PAST DATES (Historical API)
    if target_dt < yesterday:
        try:
            date_str = target_dt.strftime("%Y-%m-%d")
            url = (
                "https://archive-api.open-meteo.com/v1/archive"
                f"?latitude={lat}&longitude={lon}"
                f"&start_date={date_str}"
                f"&end_date={date_str}"
                f"&hourly=temperature_2m"
                "&timezone=Asia/Kolkata"
            )
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            times = data.get("hourly", {}).get("time", [])
            temps = data.get("hourly", {}).get("temperature_2m", [])
            
            if not times or not temps:
                return get_historical_avg_temp(target_dt)
            
            target_hour_str = target_dt.strftime("%Y-%m-%dT%H")
            for i, time_str in enumerate(times):
                if time_str.startswith(target_hour_str):
                    return temps[i]
            
            return round(sum(temps) / len(temps), 1)
            
        except:
            return get_historical_avg_temp(target_dt)
    
    # CASE 2: RECENT DATES (Forecast API)
    elif yesterday <= target_dt <= now + timedelta(days=7):
        try:
            date_str = target_dt.strftime("%Y-%m-%d")
            url = (
                "https://api.open-meteo.com/v1/forecast"
                f"?latitude={lat}&longitude={lon}"
                f"&hourly=temperature_2m"
                f"&start_date={date_str}"
                f"&end_date={date_str}"
                "&timezone=Asia/Kolkata"
            )
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            times = data.get("hourly", {}).get("time", [])
            temps = data.get("hourly", {}).get("temperature_2m", [])
            
            if not times or not temps:
                return get_historical_avg_temp(target_dt)
            
            target_hour_str = target_dt.strftime("%Y-%m-%dT%H")
            for i, time_str in enumerate(times):
                if time_str.startswith(target_hour_str):
                    return temps[i]
            
            return round(sum(temps) / len(temps), 1)
            
        except:
            return get_historical_avg_temp(target_dt)
    
    # CASE 3: FAR FUTURE DATES (Monthly Averages)
    else:
        return get_historical_avg_temp(target_dt)


def get_historical_avg_temp(target_dt):
    """Get historical average temperature based on month"""
    month_avg_temp = {
        1: 14,   # January
        2: 17,   # February
        3: 22,   # March
        4: 28,   # April
        5: 32,   # May
        6: 33,   # June
        7: 31,   # July
        8: 30,   # August
        9: 30,   # September
        10: 28,  # October
        11: 22,  # November
        12: 17   # December
    }
    return month_avg_temp.get(target_dt.month)

df['temp'] = get_temperature_delhi_final(dt)
df

,datetime,year,month,date,day,is_weekend,is_holiday,is_festival,festival_name,festival_day_last_year,festival_month_last_year,is_festival_last_year,temp
0,2025-10-20 10:00:00,2025,10,20,0,0,1,1,-1.0,31,10,0,29.3


In [22]:
# Ensure datetime is index for proper lag calculation
df = df.copy()

# If 'datetime' column exists, set it as index
if 'datetime' in df.columns:
    df = df.set_index('datetime')

# Initialize lag and moving average columns
df['moving_avg_3h'] = None
df['power_demand_1_week_ago'] = None
df['power_demand_1_year_ago'] = None

# Load historical data
historical_df = pd.read_csv('./data/dehli_energy.csv', parse_dates=['datetime'], index_col='datetime')
extended_df = historical_df.copy()

# Use mean values in case of missing lags
default_power_mean = float(extended_df['Power demand'].mean())

for dt_idx in df.index:
    # Lag: 1 week ago (168 hours)
    one_week_ago = dt_idx - pd.Timedelta(hours=168)
    try:
        df.loc[dt_idx, 'power_demand_1_week_ago'] = float(extended_df.loc[one_week_ago, 'Power demand'])
    except KeyError:
        df.loc[dt_idx, 'power_demand_1_week_ago'] = default_power_mean

    # Lag: 1 year ago (8760 hours)
    one_year_ago = dt_idx - pd.Timedelta(hours=8760)
    try:
        df.loc[dt_idx, 'power_demand_1_year_ago'] = float(extended_df.loc[one_year_ago, 'Power demand'])
    except KeyError:
        df.loc[dt_idx, 'power_demand_1_year_ago'] = default_power_mean

    # Moving average of last 3 hours from extended_df
    df.loc[dt_idx, 'moving_avg_3h'] = float(extended_df['Power demand'].tail(3).mean())

    # Append this new row to extended_df for future lag calculations
    new_row = {
        'Power demand': df.loc[dt_idx, 'Power demand'] if 'Power demand' in df.columns else default_power_mean,
        'temp': df.loc[dt_idx, 'temp'] if 'temp' in df.columns else float(extended_df['temp'].mean())
    }
    extended_df.loc[dt_idx] = new_row

df

,year,month,date,day,is_weekend,is_holiday,is_festival,festival_name,festival_day_last_year,festival_month_last_year,is_festival_last_year,temp,moving_avg_3h,power_demand_1_week_ago,power_demand_1_year_ago
datetime,,,,,,,,,,,,,,,
2025-10-20 10:00:00,2025,10,20,0,0,1,1,-1.0,31,10,0,29.3,2063.866667,3960.736469,3976.39


In [23]:
feature_cols = [
    'temp', 'year', 'month', 'date', 'day', 'is_weekend', 'is_holiday',
    'moving_avg_3h', 'power_demand_1_week_ago', 'power_demand_1_year_ago',
    'is_festival', 'festival_name', 'festival_month_last_year', 'festival_day_last_year','is_festival_last_year'
]
input_features = df[feature_cols]
# Make prediction


In [24]:
predicted_power = loaded_model.predict(input_features)[0]
print(f"🔹 Predicted Power Demand: {predicted_power:.2f} MW")

🔹 Predicted Power Demand: 35745.91 MW
